# Temporal v1 Fixed Evaluation

This notebook compares trained temporal cache-probe checkpoints on the same fixed validation set: 10 batches of 1024 samples from shard 1 of the cache-v2 data. The fixed sample removes validation-resampling noise from checkpoint comparisons.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'quant-research-workbench' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.temporal_event_model.v1.evaluate_cache_probe_checkpoints import evaluate_checkpoints

CACHE_ROOT = Path(r'D:\market-data\prepared\event_sample_cache\cache_v2_cycle_20260619_134422')
CHECKPOINT_ROOT = Path(r'D:\TradingML\runtimes\temporal_event_model\v1\cache_price_probe_laptop')
OUTPUT_DIR = Path(r'D:\TradingML\runtimes\temporal_event_model\v1\cache_price_probe_laptop_eval\fixed-shard1-10x1024')

# Leave CHECKPOINTS empty to use the three newest checkpoint_latest.pt files under CHECKPOINT_ROOT.
CHECKPOINTS = []
BATCH_SIZE = 1024
VALIDATION_BATCHES = 10
SEED = 20260621
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' else None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'device={DEVICE}')
print(f'cache_root={CACHE_ROOT}')
print(f'checkpoint_root={CHECKPOINT_ROOT}')
print(f'output_dir={OUTPUT_DIR}')

In [ ]:
if CHECKPOINTS:
    checkpoint_paths = [Path(path) for path in CHECKPOINTS]
else:
    checkpoint_paths = sorted(
        CHECKPOINT_ROOT.glob('*/checkpoints/checkpoint_latest.pt'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )[:3]

print('Checkpoints:')
for path in checkpoint_paths:
    print(' ', path)

result = evaluate_checkpoints(
    checkpoint_paths=checkpoint_paths,
    cache_root=CACHE_ROOT,
    validation_split='train',
    validation_start_shard=1,
    validation_max_shards=1,
    validation_batches=VALIDATION_BATCHES,
    batch_size=BATCH_SIZE,
    seed=SEED,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    output_dir=OUTPUT_DIR,
)
result_path = Path(result['result_path'])
print(f'wrote {result_path}')

In [ ]:
# To skip recomputation in later notebook runs, comment the previous cell and load the saved result here.
result_path = OUTPUT_DIR / 'fixed_eval_results.json'
result = json.loads(result_path.read_text(encoding='utf-8'))
class_names = result['class_names']
direction_names = result['direction_class_names']
summary = result['summary']
summary

In [ ]:
def short_name(run):
    name = run['summary']['run_name']
    return name.replace('v1-cache-probe-v20-', '').replace('-1train1val-5ep-bs512-laptop', '')

labels = [short_name(run) for run in result['runs']]
metrics = result['summary']
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, [row['accuracy_pct'] for row in metrics], width, label='5-class accuracy')
ax.bar(x, [row['macro_f1_pct'] for row in metrics], width, label='5-class macro F1')
ax.bar(x + width, [row['direction_accuracy_pct'] for row in metrics], width, label='up/down accuracy')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Percent')
ax.set_title('Fixed validation comparison')
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
def plot_confusion(ax, matrix, labels, title, normalize=True):
    matrix = np.asarray(matrix, dtype=np.float64)
    if normalize:
        denom = matrix.sum(axis=1, keepdims=True)
        values = np.divide(matrix, np.maximum(denom, 1.0)) * 100.0
        fmt = '{:.1f}'
        suffix = '%'
    else:
        values = matrix
        fmt = '{:.0f}'
        suffix = ''
    im = ax.imshow(values, cmap='Blues', vmin=0)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    threshold = values.max() * 0.55 if values.size else 0
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            text = fmt.format(values[i, j]) + suffix
            ax.text(j, i, text, ha='center', va='center', color='white' if values[i, j] > threshold else 'black', fontsize=8)
    return im

In [ ]:
for horizon_index, horizon in enumerate([128, 256]):
    fig, axes = plt.subplots(len(result['runs']), 2, figsize=(12, 4 * len(result['runs'])))
    if len(result['runs']) == 1:
        axes = np.asarray([axes])
    for row_index, run in enumerate(result['runs']):
        name = short_name(run)
        class_matrix = np.asarray(run['class_confusion'][horizon_index])
        direction_matrix = np.asarray(run['direction_confusion'][horizon_index])
        plot_confusion(axes[row_index, 0], class_matrix, class_names, f'{name} future_{horizon} 5-class')
        plot_confusion(axes[row_index, 1], direction_matrix, direction_names, f'{name} future_{horizon} up/down')
    fig.tight_layout()
    plt.show()

In [ ]:
for run in result['runs']:
    print('\n' + short_name(run))
    print(json.dumps(run['metrics'], indent=2))